# Lab03 - Nhiệm vụ 2.1 và 2.4

Notebook hỗ trợ thực hành. Chạy từ trên xuống, tự quan sát và chụp kết quả; điền báo cáo theo lần chạy của bạn. Chỉ cần môi trường CPU. Các script dùng thư viện chuẩn Python và OpenSSL.

Không tải PDF học phần lên notebook này. Không dùng mã RSA giáo khoa để bảo vệ dữ liệu thực tế.

## Chuẩn bị môi trường - chụp H01
Chạy ô sau và chụp phiên bản Python, OpenSSL. Nếu thiếu OpenSSL, xem phần xử lý lỗi trong hướng dẫn.

In [ ]:
import sys, shutil, subprocess
print("Python:", sys.version)
openssl = shutil.which("openssl")
assert openssl, "Thiếu OpenSSL: chạy !apt-get -qq update và !apt-get -qq install -y openssl"
print(subprocess.check_output([openssl, "version"], text=True))


## Tạo mã nguồn
Ô này ghi hai chương trình vào thư mục làm việc. Có thể mở tệp ở bảng Files bên trái để đọc các hàm trước khi chạy.

In [ ]:
from pathlib import Path
import os
folder = Path("/content/Lab03_RSA_X509")
folder.mkdir(exist_ok=True)
(folder / 'task21.py').write_text('"""Lab 03, nhiệm vụ 2.1. RSA giáo khoa, chỉ dùng thư viện chuẩn Python.\n\nVí dụ: python task21.py keys / number / text / decrypt / random / all\nQuy ước: UTF-8, số nguyên little-endian, khối bản mã có độ dài cố định.\nRSA không padding này chỉ dùng để học, không dùng bảo vệ dữ liệu thật.\n"""\nimport argparse\nimport base64\nimport math\nfrom pathlib import Path\nimport secrets\nimport sys\nimport time\n\nPARAMETERS = [\n    (11, 17, 7),\n    (20079993872842322116151219, 676717145751736242170789, 17),\n    (int(\'F7E75FDC469067FFDC4E847C51F452DF\', 16),\n     int(\'E85CED54AF57E53E092113E62F436F4F\', 16), int(\'0D88C3\', 16)),\n]\nGIVEN = [\n    (\'Base64\', \'raUcesUlOkx/8ZhgodMoo0Uu18sC20yXlQFevSu7W/FDxIy0YRHMyXcHdD9PBvIT2aUft5fCQEGomiVVPv4I\'),\n    (\'HEX\', \'C87F570FC4F699CEC24020C6F54221ABAB2CE0C3\'),\n    (\'Base64\', \'Z2BUSkJcg0w4XEpgm0JcMExEQmBlVH6dYEpNTHpMHptMQ7NgTHlgQrNMQ2BKTQ==\'),\n    (\'BIN\', \'001010000001010011111111101101110010111011001010111011000110011\'\n            \'110111111001111110110100011001111001100001001010001010100111101\'\n            \'010100110011101110111011110101101100000100\'),\n]\nMESSAGE = \'The University of Information Technology\'\n\n\ndef probable_prime(n, rounds=40):\n    """Miller-Rabin với cơ sở ngẫu nhiên; sai số <= 4**(-rounds)."""\n    if n < 2:\n        return False\n    for p in (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37):\n        if n % p == 0:\n            return n == p\n    r, t = 0, n - 1\n    while t % 2 == 0:\n        r, t = r + 1, t // 2\n    for _ in range(rounds):\n        a = secrets.randbelow(n - 3) + 2\n        x = pow(a, t, n)\n        if x in (1, n - 1):\n            continue\n        for _ in range(r - 1):\n            x = pow(x, 2, n)\n            if x == n - 1:\n                break\n        else:\n            return False\n    return True\n\n\ndef random_prime(bits):\n    while True:\n        n = secrets.randbits(bits) | (1 << (bits - 1)) | 1\n        if probable_prime(n):\n            return n\n\n\ndef make_key(p=None, q=None, e=None, bits=2048):\n    """Nhận đủ p,q hoặc tự sinh cả hai; e mặc định là 65537."""\n    if (p is None) != (q is None):\n        raise ValueError(\'Cung cấp đồng thời p và q, hoặc bỏ cả hai.\')\n    e = 65537 if e is None else e\n    if p is None:\n        if bits < 512 or e <= 1 or e % 2 == 0:\n            raise ValueError(\'Khóa tự sinh: bits >= 512 và e lẻ > 1.\')\n        if e >= (1 << (bits - 1)):\n            raise ValueError(\'e quá lớn so với kích thước khóa được chọn.\')\n        while True:\n            p, q = random_prime(bits // 2), random_prime(bits - bits // 2)\n            phi = (p - 1) * (q - 1)\n            if p != q and (p * q).bit_length() == bits and math.gcd(e, phi) == 1 and e < phi:\n                break\n    elif p == q or not probable_prime(p) or not probable_prime(q):\n        raise ValueError(\'p và q phải là hai số nguyên tố khác nhau.\')\n    n, phi = p * q, (p - 1) * (q - 1)\n    if not 1 < e < phi or math.gcd(e, phi) != 1:\n        raise ValueError(\'Cần 1 < e < phi(n) và gcd(e, phi(n)) = 1.\')\n    return dict(p=p, q=q, n=n, phi=phi, e=e, d=pow(e, -1, phi))\n\n\ndef rsa_number(value, exponent, modulus):\n    if not 0 <= value < modulus:\n        raise ValueError(\'Mỗi khối phải nằm trong khoảng [0, n-1].\')\n    return pow(value, exponent, modulus)\n\n\ndef sizes(n):\n    # k là kích thước khối bản mã. b bảo đảm 256**b <= n.\n    # Riêng khóa 1 quá nhỏ cho byte bất kỳ: dùng từng byte và kiểm tra < n.\n    return max(1, (n.bit_length() - 1) // 8), (n.bit_length() + 7) // 8\n\n\ndef encrypt_text(text, key):\n    raw = text.encode(\'utf-8\')\n    b, k = sizes(key[\'n\'])\n    blocks = []\n    for start in range(0, len(raw), b):\n        m = int.from_bytes(raw[start:start + b], \'little\')\n        c = rsa_number(m, key[\'e\'], key[\'n\'])\n        blocks.append(c.to_bytes(k, \'little\'))\n    return b\'\'.join(blocks), len(raw)\n\n\ndef decrypt_text(cipher, original_length, key):\n    b, k = sizes(key[\'n\'])\n    if len(cipher) % k or len(cipher) // k != (original_length + b - 1) // b:\n        raise ValueError(\'Độ dài bản mã không khớp với số khối.\')\n    result = bytearray()\n    for start in range(0, len(cipher), k):\n        c = int.from_bytes(cipher[start:start + k], \'little\')\n        m = rsa_number(c, key[\'d\'], key[\'n\'])\n        width = min(b, original_length - len(result))\n        result.extend(m.to_bytes(width, \'little\'))\n    return result.decode(\'utf-8\')\n\n\ndef decode_given(kind, value):\n    if kind == \'Base64\':\n        return base64.b64decode(value, validate=True)\n    if kind == \'HEX\':\n        return bytes.fromhex(value)\n    # BIN là chuỗi bit của dãy byte, giữ cả các số 0 ở đầu.\n    if len(value) % 8:\n        raise ValueError(\'Chuỗi bit phải có độ dài là bội của 8.\')\n    return bytes(int(value[i:i+8], 2) for i in range(0, len(value), 8))\n\n\ndef candidate_plaintext(raw, key):\n    """Thử định dạng của đề; khối cuối có thể bỏ byte 00 cao nhất.\n\n    Đề không cho độ dài gốc. Giả sử văn bản ASCII không chứa NUL cuối.\n    Các khối không phải cuối phục hồi đúng b byte; khối cuối dài tối thiểu.\n    """\n    b, k = sizes(key[\'n\'])\n    result = bytearray()\n    block_sizes = []\n    roundtrip = True\n    for start in range(0, len(raw), k):\n        part = raw[start:start+k]\n        c = int.from_bytes(part, \'little\')\n        m = rsa_number(c, key[\'d\'], key[\'n\'])\n        width = b if start+k < len(raw) else max(1, (m.bit_length()+7)//8)\n        if width > b:\n            raise ValueError(\'Khối bản rõ quá dài.\')\n        result.extend(m.to_bytes(width, \'little\'))\n        roundtrip &= rsa_number(m, key[\'e\'], key[\'n\']).to_bytes(len(part), \'little\') == part\n        block_sizes.append(len(part))\n    if not result or not all(32 <= x < 127 for x in result):\n        raise ValueError(\'Không tạo thành văn bản ASCII đọc được.\')\n    return result.decode(\'ascii\'), block_sizes, roundtrip\n\n\ndef demo(part, bits=2048):\n    keys = [make_key(*params) for params in PARAMETERS]\n    if part in (\'keys\', \'all\'):\n        print(\'2.1.1 - TẠO BA CẶP KHÓA\')\n        for i, key in enumerate(keys, 1):\n            print(f\'\\nKhóa {i}:\')\n            for name in (\'n\', \'phi\', \'e\', \'d\'):\n                print(f\'{name} = {key[name]}\')\n            print(\'PU = (e, n); PR = (d, n)\')\n            print(\'Số bit n =\', key[\'n\'].bit_length())\n            print(\'(e*d) mod phi =\', key[\'e\']*key[\'d\'] % key[\'phi\'])\n    if part in (\'number\', \'all\'):\n        key, m = keys[0], 5\n        c = rsa_number(m, key[\'e\'], key[\'n\'])\n        s = rsa_number(m, key[\'d\'], key[\'n\'])\n        print(\'\\n2.1.2 - BẢN RÕ M = 5, KHÓA 1\')\n        print(\'Bảo mật: M =\', m, \'-> C =\', c, \'-> M =\', rsa_number(c, key[\'d\'], key[\'n\']))\n        print(\'Xác thực theo mô hình đề: M =\', m, \'-> S =\', s, \'-> M =\', rsa_number(s, key[\'e\'], key[\'n\']))\n    if part in (\'text\', \'all\'):\n        print(\'\\n2.1.3 - MÃ HÓA CHUỖI, XUẤT BASE64\')\n        print(\'Bản rõ:\', MESSAGE)\n        for i, key in enumerate(keys, 1):\n            cipher, length = encrypt_text(MESSAGE, key)\n            recovered = decrypt_text(cipher, length, key)\n            print(f\'\\nKhóa {i}; khối rõ/mã (byte) = {sizes(key["n"])}; độ dài gốc = {length}\')\n            print(\'Base64 =\', base64.b64encode(cipher).decode())\n            print(\'Giải mã =\', recovered)\n            print(\'Khôi phục đúng =\', recovered == MESSAGE)\n    if part in (\'decrypt\', \'all\'):\n        print(\'\\n2.1.4 - GIẢI BỐN BẢN MÃ CHO SẴN\')\n        for index, (kind, value) in enumerate(GIVEN, 1):\n            raw = decode_given(kind, value)\n            print(f\'\\nBản mã {index}; {kind}; {len(raw)} byte\')\n            found = False\n            for ki, key in enumerate(keys, 1):\n                try:\n                    plain, widths, same = candidate_plaintext(raw, key)\n                except (ValueError, OverflowError, UnicodeDecodeError):\n                    continue\n                found = True\n                print(f\'Khóa {ki}; little-endian; khối mã = {widths}\')\n                print(\'Bản rõ =\', plain)\n                print(\'Mã hóa lại trùng bản mã =\', same)\n            if not found:\n                print(\'Không có ứng viên phù hợp với quy ước đã thử.\')\n    if part in (\'random\', \'all\'):\n        print(f\'\\nBỔ SUNG - SINH KHÓA NGẪU NHIÊN {bits} BIT\', flush=True)\n        start = time.perf_counter()\n        key = make_key(bits=bits)\n        for name in (\'n\', \'e\', \'d\'):\n            print(f\'{name} = {key[name]}\')\n        print(\'Số bit n =\', key[\'n\'].bit_length())\n        print(\'(e*d) mod phi =\', key[\'e\']*key[\'d\'] % key[\'phi\'])\n        sample = \'Thực hành RSA - An toàn mạng máy tính\'\n        cipher, length = encrypt_text(sample, key)\n        print(\'Bản rõ =\', sample)\n        print(\'Base64 =\', base64.b64encode(cipher).decode())\n        print(\'Giải mã =\', decrypt_text(cipher, length, key))\n        print(\'Khôi phục đúng =\', decrypt_text(cipher, length, key) == sample)\n        print(f\'Thời gian sinh khóa và thử: {time.perf_counter()-start:.2f} giây\')\n\n\nif __name__ == \'__main__\':\n    sys.stdout.reconfigure(encoding=\'utf-8\')\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'part\', choices=[\'keys\', \'number\', \'text\', \'decrypt\', \'random\', \'all\'], nargs=\'?\', default=\'all\')\n    parser.add_argument(\'--bits\', type=int, default=2048)\n    args = parser.parse_args()\n    demo(args.part, args.bits)\n', encoding="utf-8")
(folder / 'task24.py').write_text('"""Lab 03, nhiệm vụ 2.4: trích xuất X.509, tự xác minh RSA/SHA-256.\n\nChỉ cần Python chuẩn và OpenSSL. Không gọi hàm verify của thư viện RSA.\nLệnh: python task24.py fetch --host www.amazon.com\n      python task24.py inspect\n      python task24.py verify\n"""\nimport argparse\nimport base64\nfrom datetime import datetime, timezone\nimport hashlib\nimport hmac\nimport json\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport sys\n\nBASE = Path(__file__).resolve().parent\nCERTS = BASE / \'certs\'\nSHA256_RSA_OID = bytes.fromhex(\'2a864886f70d01010b\')\nDIGEST_INFO_PREFIX = bytes.fromhex(\'3031300d060960864801650304020105000420\')\n\n\ndef openssl_path():\n    native_git = Path(r\'C:\\Program Files\\Git\\mingw64\\bin\\openssl.exe\')\n    found = str(native_git) if native_git.exists() else shutil.which(\'openssl\')\n    if not found:\n        raise RuntimeError(\'Không tìm thấy OpenSSL. Dùng Git for Windows hoặc Colab.\')\n    return found\n\n\ndef openssl(*args, data=None, timeout=30, require_success=True):\n    result = subprocess.run([openssl_path(), *map(str, args)], input=data,\n                            stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=timeout)\n    if require_success and result.returncode:\n        raise RuntimeError(result.stderr.decode(errors=\'replace\').strip())\n    return result\n\n\ndef tlv(data, offset):\n    """Đọc một trường DER đơn giản: trả tag, đầu nội dung, cuối trường."""\n    if offset + 2 > len(data):\n        raise ValueError(\'DER bị thiếu dữ liệu.\')\n    tag, length, pos = data[offset], data[offset+1], offset+2\n    if tag & 31 == 31:\n        raise ValueError(\'Không hỗ trợ tag ASN.1 nhiều byte trong ví dụ này.\')\n    if length & 128:\n        count = length & 127\n        if not count or count > 4 or pos+count > len(data):\n            raise ValueError(\'Độ dài DER không hợp lệ.\')\n        length = int.from_bytes(data[pos:pos+count], \'big\')\n        pos += count\n    end = pos + length\n    if end > len(data):\n        raise ValueError(\'Trường DER vượt quá kích thước tệp.\')\n    return tag, pos, end\n\n\ndef fields(data, start, end):\n    result = []\n    while start < end:\n        tag, content, next_offset = tlv(data, start)\n        if next_offset > end:\n            raise ValueError(\'Trường con vượt quá trường cha.\')\n        result.append((start, tag, content, next_offset))\n        start = next_offset\n    if start != end:\n        raise ValueError(\'DER không hợp lệ.\')\n    return result\n\n\ndef certificate_parts(pem):\n    match = re.search(rb\'-----BEGIN CERTIFICATE-----\\s*(.*?)\\s*-----END CERTIFICATE-----\', pem, re.S)\n    if not match:\n        raise ValueError(\'Không tìm thấy chứng chỉ PEM.\')\n    der = base64.b64decode(re.sub(rb\'\\s+\', b\'\', match.group(1)), validate=True)\n    tag, start, end = tlv(der, 0)\n    if tag != 0x30 or end != len(der):\n        raise ValueError(\'Chứng chỉ không phải một SEQUENCE DER hợp lệ.\')\n    children = fields(der, start, end)\n    if len(children) != 3:\n        raise ValueError(\'Certificate phải có ba trường.\')\n    tbs, algorithm, signature = children\n    if (tbs[1], algorithm[1], signature[1]) != (0x30, 0x30, 0x03):\n        raise ValueError(\'Cấu trúc chứng chỉ không hợp lệ.\')\n    alg_children = fields(der, algorithm[2], algorithm[3])\n    oid = alg_children[0]\n    if oid[1] != 0x06 or der[oid[2]:oid[3]] != SHA256_RSA_OID:\n        raise ValueError(\'Cần chứng chỉ sha256WithRSAEncryption (PKCS#1 v1.5). Hãy chọn website khác.\')\n    tbs_children = fields(der, tbs[2], tbs[3])\n    inner_algorithm = tbs_children[2 if tbs_children[0][1] == 0xa0 else 1]\n    if der[inner_algorithm[0]:inner_algorithm[3]] != der[algorithm[0]:algorithm[3]]:\n        raise ValueError(\'Thuật toán chữ ký bên trong và bên ngoài không khớp.\')\n    if der[signature[2]] != 0:\n        raise ValueError(\'BIT STRING chữ ký phải có 0 bit không sử dụng.\')\n    return dict(der=der, tbs=der[tbs[0]:tbs[3]],\n                signature=der[signature[2]+1:signature[3]],\n                offset=tbs[0], header=tbs[2]-tbs[0], content_length=tbs[3]-tbs[2])\n\n\ndef distinguished_name(pem, field):\n    value = openssl(\'x509\', \'-noout\', \'-nameopt\', \'RFC2253\', \'-\'+field, data=pem).stdout.decode().strip()\n    return value.split(\'=\', 1)[1].strip()\n\n\ndef issuer_key(pem):\n    modulus = openssl(\'x509\', \'-noout\', \'-modulus\', data=pem).stdout.decode().strip()\n    if not re.fullmatch(r\'Modulus=[0-9A-Fa-f]+\', modulus):\n        raise ValueError(\'Khóa công khai của CA không phải RSA.\')\n    public_pem = openssl(\'x509\', \'-noout\', \'-pubkey\', data=pem).stdout\n    key_text = openssl(\'pkey\', \'-pubin\', \'-text\', \'-noout\', data=public_pem).stdout.decode()\n    exponent = re.search(r\'Exponent:\\s*(\\d+)\', key_text, re.I)\n    if not exponent:\n        raise ValueError(\'Không đọc được số mũ e của khóa CA.\')\n    return int(modulus.split(\'=\')[1], 16), int(exponent.group(1)), public_pem\n\n\ndef fetch(host):\n    if not re.fullmatch(r\'[A-Za-z0-9](?:[A-Za-z0-9.-]*[A-Za-z0-9])?\', host):\n        raise ValueError(\'Chỉ nhập tên miền, không có https:// hoặc đường dẫn.\')\n    command = [\'s_client\', \'-connect\', host+\':443\', \'-servername\', host, \'-showcerts\']\n    response = openssl(*command, data=b\'\', require_success=False)\n    chain = re.findall(rb\'-----BEGIN CERTIFICATE-----.*?-----END CERTIFICATE-----\', response.stdout, re.S)\n    if not chain:\n        raise RuntimeError(\'Không tải được chứng chỉ. Kiểm tra Internet/DNS.\\n\' + response.stderr.decode(errors=\'replace\'))\n    leaf = chain[0]+b\'\\n\'\n    certificate_parts(leaf)  # Từ chối ECDSA/PSS trước khi ghi đè dữ liệu.\n    issuer = distinguished_name(leaf, \'issuer\')\n    matches = [c+b\'\\n\' for c in chain[1:] if distinguished_name(c, \'subject\') == issuer]\n    if not matches:\n        raise ValueError(\'Máy chủ chưa gửi chứng chỉ CA phù hợp. Hãy chọn website khác.\')\n    selected = None\n    for candidate in matches:\n        try:\n            n, e, _ = issuer_key(candidate)\n            part = certificate_parts(leaf)\n            if check_signature(part[\'signature\'], part[\'tbs\'], n, e)[0]:\n                selected = candidate\n                break\n        except (ValueError, RuntimeError):\n            continue\n    if selected is None:\n        raise ValueError(\'Không tìm được CA RSA xác minh được chữ ký máy chủ.\')\n    CERTS.mkdir(exist_ok=True)\n    (CERTS/\'chain.txt\').write_bytes(response.stdout + b\'\\n\' + response.stderr)\n    (CERTS/\'c0.pem\').write_bytes(leaf)\n    (CERTS/\'c1.pem\').write_bytes(selected)\n    info = dict(host=host, fetched_at_utc=datetime.now(timezone.utc).isoformat(),\n                command=\'openssl \'+\' \'.join(command), chain_certificates=len(chain))\n    (CERTS/\'download.json\').write_text(json.dumps(info, indent=2), encoding=\'utf-8\')\n    print(\'Website:\', host)\n    print(\'Số chứng chỉ máy chủ gửi:\', len(chain))\n    print(\'c0 subject:\', distinguished_name(leaf, \'subject\'))\n    print(\'c0 issuer :\', issuer)\n    print(\'c1 subject:\', distinguished_name(selected, \'subject\'))\n    print(\'Thuật toán chữ ký: sha256WithRSAEncryption\')\n    print(\'Đã lưu c0.pem, c1.pem, chain.txt và download.json vào\', CERTS)\n\n\ndef inspect():\n    leaf, issuer = (CERTS/\'c0.pem\').read_bytes(), (CERTS/\'c1.pem\').read_bytes()\n    part = certificate_parts(leaf)\n    n, e, public_pem = issuer_key(issuer)\n    for name, data in [(\'c0.der\', part[\'der\']), (\'c0_body.bin\', part[\'tbs\']),\n                       (\'signature.bin\', part[\'signature\']), (\'ca_public.pem\', public_pem)]:\n        (CERTS/name).write_bytes(data)\n    (CERTS/\'signature.hex\').write_text(part[\'signature\'].hex(), encoding=\'ascii\')\n    asn1 = openssl(\'asn1parse\', \'-i\', \'-in\', CERTS/\'c0.pem\').stdout\n    (CERTS/\'asn1.txt\').write_bytes(asn1)\n    for stem in (\'c0\', \'c1\'):\n        (CERTS/(stem+\'_text.txt\')).write_bytes(openssl(\'x509\', \'-in\', CERTS/(stem+\'.pem\'), \'-text\', \'-noout\').stdout)\n    # Trích bằng OpenSSL, rồi so với bộ đọc DER để phát hiện sai offset/phạm vi.\n    openssl(\'asn1parse\', \'-in\', CERTS/\'c0.pem\', \'-strparse\', str(part[\'offset\']),\n            \'-out\', CERTS/\'c0_body_openssl.bin\', \'-noout\')\n    same = (CERTS/\'c0_body_openssl.bin\').read_bytes() == part[\'tbs\']\n    if not same:\n        raise ValueError(\'Phần TBS do OpenSSL trích không khớp bộ đọc DER.\')\n    info = dict(n=str(n), n_hex=format(n, \'X\'), e=e, n_bits=n.bit_length(),\n                tbs_offset=part[\'offset\'], tbs_header=part[\'header\'],\n                tbs_content_length=part[\'content_length\'], tbs_length=len(part[\'tbs\']),\n                signature_length=len(part[\'signature\']), sha256=hashlib.sha256(part[\'tbs\']).hexdigest())\n    (CERTS/\'extracted.json\').write_text(json.dumps(info, indent=2), encoding=\'utf-8\')\n    print(\'c0 subject:\', distinguished_name(leaf, \'subject\'))\n    print(\'c0 issuer :\', distinguished_name(leaf, \'issuer\'))\n    print(\'c1 subject:\', distinguished_name(issuer, \'subject\'))\n    print(\'\\nKHÓA CÔNG KHAI CỦA CA\')\n    print(\'n (HEX) =\', format(n, \'X\'))\n    print(\'e =\', e, \'; số bit n =\', n.bit_length())\n    print(\'\\nCHỮ KÝ CỦA CA TRÊN c0 (HEX)\')\n    print(part[\'signature\'].hex())\n    print(\'Độ dài chữ ký =\', len(part[\'signature\']), \'byte\')\n    print(\'\\nPHẦN TBSCertificate\')\n    print(\'Offset =\', part[\'offset\'], \'; hl =\', part[\'header\'], \'; l =\', part[\'content_length\'])\n    print(\'Độ dài TBS = hl + l =\', len(part[\'tbs\']), \'byte\')\n    print(\'TBS từ OpenSSL khớp bộ đọc DER =\', same)\n    print(\'SHA-256(TBS) =\', info[\'sha256\'])\n    print(\'Các tệp đã lưu trong\', CERTS)\n\n\ndef check_signature(signature, tbs, n, e):\n    k = (n.bit_length()+7)//8\n    s = int.from_bytes(signature, \'big\')\n    if len(signature) != k or s >= n:\n        return False, b\'\', b\'\'\n    # Phép RSA xác minh yêu cầu của bài lab.\n    recovered = pow(s, e, n).to_bytes(k, \'big\')\n    digest_info = DIGEST_INFO_PREFIX + hashlib.sha256(tbs).digest()\n    padding_length = k - len(digest_info) - 3\n    if padding_length < 8:\n        return False, recovered, b\'\'\n    expected = b\'\\x00\\x01\' + b\'\\xff\'*padding_length + b\'\\x00\' + digest_info\n    return hmac.compare_digest(recovered, expected), recovered, expected\n\n\ndef verify():\n    leaf, issuer = (CERTS/\'c0.pem\').read_bytes(), (CERTS/\'c1.pem\').read_bytes()\n    part = certificate_parts(leaf)\n    n, e, _ = issuer_key(issuer)\n    names_match = distinguished_name(leaf, \'issuer\') == distinguished_name(issuer, \'subject\')\n    valid, recovered, expected = check_signature(part[\'signature\'], part[\'tbs\'], n, e)\n    modified = bytearray(part[\'tbs\'])\n    modified[-1] ^= 1\n    tampered_valid, _, _ = check_signature(part[\'signature\'], bytes(modified), n, e)\n    print(\'Thuật toán: sha256WithRSAEncryption / PKCS#1 v1.5\')\n    print(\'c0.issuer == c1.subject:\', names_match)\n    print(\'Số bit khóa công khai CA:\', n.bit_length())\n    print(\'EM = S^e mod n, độ dài:\', len(recovered), \'byte\')\n    print(\'Đầu EM:\', recovered[:16].hex())\n    print(\'SHA-256 tính từ TBS:\', hashlib.sha256(part[\'tbs\']).hexdigest())\n    print(\'32 byte cuối EM   :\', recovered[-32:].hex())\n    print(\'Khớp TOÀN BỘ padding + DigestInfo + hash:\', valid)\n    print(\'CHỮ KÝ HỢP LỆ:\', valid and names_match)\n    print(\'Sau khi đổi 1 bit của TBS, chữ ký hợp lệ:\', tampered_valid)\n    print(\'Phạm vi: kiểm tra chữ ký; chưa kiểm tra đầy đủ chuỗi tin cậy, tên miền, thời hạn và thu hồi.\')\n    (CERTS/\'verification.json\').write_text(json.dumps(dict(signature_valid=valid,\n        issuer_name_matches=names_match, tampered_signature_valid=tampered_valid,\n        sha256=hashlib.sha256(part[\'tbs\']).hexdigest(), recovered_em=recovered.hex(),\n        expected_em=expected.hex()), indent=2), encoding=\'utf-8\')\n    if not valid or not names_match or tampered_valid:\n        raise ValueError(\'Kiểm tra không đạt.\')\n\n\nif __name__ == \'__main__\':\n    sys.stdout.reconfigure(encoding=\'utf-8\')\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'action\', choices=[\'fetch\', \'inspect\', \'verify\'])\n    parser.add_argument(\'--host\', default=\'www.amazon.com\')\n    args = parser.parse_args()\n    try:\n        if args.action == \'fetch\':\n            fetch(args.host)\n        elif args.action == \'inspect\':\n            inspect()\n        else:\n            verify()\n    except (ValueError, RuntimeError, OSError, subprocess.TimeoutExpired) as exc:\n        print(\'LỖI:\', exc)\n        sys.exit(1)\n', encoding="utf-8")
os.chdir(folder)
print("Thư mục làm việc:", folder)


## 2.1.1 - Tạo khóa

Chụp H02a/H02b: n, phi, e, d của ba khóa và kiểm tra nghịch đảo modulo.

In [ ]:
!python task21.py keys


## 2.1.2 - Bản rõ M=5

Chụp H03: hai kết quả 5 → 146 → 5 và 5 → 180 → 5.

In [ ]:
!python task21.py number


## 2.1.3 - Mã hóa chuỗi

Chụp H04: ba Base64 và kết quả khôi phục đúng. Quy ước UTF-8, little-endian, khối rõ/mã 1/1, 20/21, 31/32 byte.

In [ ]:
!python task21.py text


## 2.1.4 - Giải bốn bản mã

Chụp H05: bốn bản rõ, số khóa và kết quả mã hóa lại trùng bản mã.

In [ ]:
!python task21.py decrypt


## 2.1 - Khóa tự sinh

Chụp H06: số bit khóa, e, kiểm tra nghịch đảo, khôi phục chuỗi tiếng Việt và thời gian. Có thể đổi 2048 thành 4096; không gọi đó là giới hạn lớn nhất tuyệt đối.

In [ ]:
!python task21.py random --bits 2048


## 2.4 - Lấy chứng chỉ

Chụp H07: website, subject/issuer, thuật toán. Website có thể đổi chứng chỉ; nếu báo không đúng thuật toán, chọn máy chủ khác có sha256WithRSAEncryption.

In [ ]:
!python task24.py fetch --host www.amazon.com


## 2.4 - Khóa CA

Chụp H08: n, e và số bit của khóa CA.

In [ ]:
!openssl x509 -in certs/c1.pem -noout -modulus
!openssl x509 -in certs/c1.pem -text -noout | grep -E "Public Key Algorithm|Public-Key|Exponent"


## 2.4 - Xem chữ ký

Chụp H09: Signature Algorithm và Signature Value của c0.

In [ ]:
!openssl x509 -in certs/c0.pem -text -noout | tail -24


## 2.4 - Trích TBS và chữ ký

Chụp H10a/H10b: offset, hl, l của TBS, độ dài và hai hash trùng nhau. Không mặc định offset luôn bằng 4.

In [ ]:
!python task24.py inspect
!openssl asn1parse -i -in certs/c0.pem | head -12
!sha256sum certs/c0_body.bin certs/c0_body_openssl.bin


## 2.4 - Xác minh RSA

Chụp H11: khớp toàn bộ PKCS#1 v1.5, chữ ký True, đổi một bit False. Đây là kiểm tra chữ ký, chưa phải xác minh đầy đủ độ tin cậy TLS.

In [ ]:
!python task24.py verify


## Tải mã nguồn và dữ liệu thực nghiệm
Chạy ô này sau khi hoàn tất. ZIP chứa mã nguồn và dữ liệu chứng chỉ; tải riêng notebook có kết quả qua File > Download > Download .ipynb. Lưu trước khi phiên Colab hết hạn. Chèn ảnh và điền thông tin nhóm vào báo cáo trước khi xuất PDF.

In [ ]:
import zipfile
from google.colab import files
zip_path = Path("/content/Lab03_Resource_2_1_2_4.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ("task21.py", "task24.py"):
        z.write(folder/name, name)
    for path in sorted((folder/"certs").glob("*")):
        if path.is_file():
            z.write(path, str(path.relative_to(folder)))
files.download(str(zip_path))


## Nguồn tham khảo
- PDF Lab03 do giảng viên cung cấp, nhiệm vụ 2.1 và 2.4.
- https://docs.python.org/3/library/functions.html#pow
- https://docs.openssl.org/3.0/man1/openssl-s_client/
- https://docs.openssl.org/3.5/man1/openssl-asn1parse/
- https://www.rfc-editor.org/rfc/rfc8017.html

Đọc, hiểu và điều chỉnh mã theo cách thực hiện của nhóm; ghi nguồn hỗ trợ theo quy định học phần.